# Baseline Analysis: Alpha=0.1, Seed=42, 3 Strategies

Analisis awal dari 3 eksperimen yang sudah dijalankan untuk memahami pola fairness metrics sebelum 54-experiment full grid.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Paths
RESULTS_DIR = Path("results")
DATASET = "mnist"
ALPHA = 0.1
SEED = 42
STRATEGIES = ["random", "performance", "fairness"]

print("="*60)
print(f"BASELINE ANALYSIS: {DATASET} | alpha={ALPHA} | seed={SEED}")
print("="*60)
print(f"Strategies: {STRATEGIES}\n")

# Load semua hasil
experiments = {}
for strategy in STRATEGIES:
    exp_id = f"{strategy}_{DATASET}_a{ALPHA}_s{SEED}"
    exp_dir = RESULTS_DIR / exp_id
    
    if not exp_dir.exists():
        print(f"WARNING: {exp_id} tidak ditemukan")
        continue
    
    with open(exp_dir / "final_metrics.json") as f:
        final_metrics = json.load(f)
    with open(exp_dir / "metrics_per_round.json") as f:
        per_round = json.load(f)
    with open(exp_dir / "participation_log.json") as f:
        participation = json.load(f)
    
    experiments[strategy] = {
        "final_metrics": final_metrics,
        "per_round": per_round,
        "participation": participation,
    }

print(f"Loaded {len(experiments)} eksperimen")

In [ ]:
# Cell 2: Visualisasi Partisi Dirichlet
import sys
sys.path.insert(0, str(Path.cwd()))

from src.data.partitioner import load_partition_info

partition_info = load_partition_info(DATASET, ALPHA, SEED)

fig, axes = plt.subplots(2, 5, figsize=(16, 8))
axes = axes.flatten()

for i, info in enumerate(partition_info):
    ax = axes[i]
    client_id = info["client_id"]
    class_dist = info["class_distribution"]
    total = info["total_samples"]
    
    classes = sorted(int(k) for k in class_dist.keys())
    counts = [class_dist[str(c)] for c in classes]
    
    ax.bar(classes, counts, color='steelblue', edgecolor='black')
    ax.set_title(f"Client {client_id} (n={total})", fontweight='bold')
    ax.set_xlabel("Class")
    ax.set_ylabel("Count")
    ax.set_xticks(range(10))
    ax.grid(True, alpha=0.3)
    
    dom_cls = info["dominant_class"]
    dom_pct = info["dominant_class_pct"]
    ax.text(0.98, 0.97, f"Dom: {dom_cls}\n({dom_pct:.1f}%)",
            transform=ax.transAxes, ha='right', va='top',
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3))

plt.tight_layout()
plt.savefig("results/00_partition_distribution.png", dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: results/00_partition_distribution.png")

In [ ]:
# Cell 3: Accuracy Progression
fig, ax = plt.subplots(figsize=(12, 6))

for strategy, data in experiments.items():
    per_round = data["per_round"]
    rounds = [r["round"] for r in per_round]
    accs = [r["global_accuracy"] for r in per_round]
    
    ax.plot(rounds, accs, marker='o', label=strategy, linewidth=2, markersize=6)

ax.axhline(y=85.0, color='red', linestyle='--', linewidth=1, alpha=0.5, label="Target (85%)")
ax.set_xlabel("Communication Round", fontweight='bold')
ax.set_ylabel("Global Test Accuracy (%)", fontweight='bold')
ax.set_title(f"Accuracy Progression: {DATASET} alpha={ALPHA}", fontweight='bold', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 105])

plt.tight_layout()
plt.savefig("results/01_accuracy_progression.png", dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: results/01_accuracy_progression.png")

In [ ]:
# Cell 4: Fairness Metrics Comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics_data = []
for strategy, data in experiments.items():
    metrics = data["final_metrics"]
    metrics_data.append({
        "Strategy": strategy,
        "B1 (Var)": metrics["B1_accuracy_variance"],
        "B2 (Gini)": metrics["B2_gini_coefficient"],
        "B3 (Part)": metrics["B3_participation_fairness"],
    })

df = pd.DataFrame(metrics_data)

for idx, col in enumerate(["B1 (Var)", "B2 (Gini)", "B3 (Part)"]):
    ax = axes[idx]
    values = df[col].values
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    
    bars = ax.bar(STRATEGIES, values, color=colors, edgecolor='black', linewidth=1.5)
    ax.set_ylabel(col, fontweight='bold')
    ax.set_title(col, fontweight='bold', fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig("results/02_fairness_metrics.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nFairness Metrics Comparison:")
print(df.to_string(index=False))
print("\n✓ Saved: results/02_fairness_metrics.png")

In [ ]:
# Cell 5: Participation Fairness
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (strategy, data) in enumerate(experiments.items()):
    ax = axes[idx]
    final_counts = data["participation"]["final_counts"]
    
    client_ids = sorted(int(k) for k in final_counts.keys())
    counts = [final_counts[str(cid)] for cid in client_ids]
    
    bars = ax.bar(client_ids, counts, color='steelblue', edgecolor='black')
    ax.set_xlabel("Client ID", fontweight='bold')
    ax.set_ylabel("Selection Count", fontweight='bold')
    ax.set_title(f"{strategy.capitalize()}", fontweight='bold', fontsize=12)
    ax.set_xticks(range(10))
    ax.grid(True, alpha=0.3, axis='y')
    
    for bar, cnt in zip(bars, counts):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(cnt)}', ha='center', va='bottom', fontsize=9)
    
    std_dev = np.std(counts)
    mean_count = np.mean(counts)
    ax.text(0.98, 0.98, f"μ={mean_count:.1f}\nσ={std_dev:.2f}",
            transform=ax.transAxes, ha='right', va='top',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))

plt.tight_layout()
plt.savefig("results/03_participation_fairness.png", dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: results/03_participation_fairness.png")

In [ ]:
# Cell 6: Summary Table
summary_data = []
for strategy, data in experiments.items():
    metrics = data["final_metrics"]
    final_counts = data["participation"]["final_counts"]
    counts = [final_counts[str(i)] for i in range(10)]
    
    summary_data.append({
        "Strategy": strategy,
        "A1 (Acc %)": f"{metrics['A1_global_accuracy']:.2f}",
        "A2 (Rounds)": metrics["A2_rounds_to_target"],
        "B1 (Var)": f"{metrics['B1_accuracy_variance']:.6f}",
        "B2 (Gini)": f"{metrics['B2_gini_coefficient']:.6f}",
        "B3 (Part Fair)": f"{metrics['B3_participation_fairness']:.4f}",
        "Min/Max Count": f"{min(counts)}/{max(counts)}",
    })

df_summary = pd.DataFrame(summary_data)
print("\n" + "="*80)
print("COMPREHENSIVE SUMMARY TABLE")
print("="*80)
print(df_summary.to_string(index=False))

df_summary.to_csv("results/summary_baseline.csv", index=False)
print("\n✓ Saved: results/summary_baseline.csv")

## Early Insights dan Next Steps

### Hasil Awal (Alpha=0.1, Seed=42):

1. **Global Performance (A1, A2)**
   - Semua strategi mencapai target accuracy
   - Pada MNIST (mudah), selection strategy tidak berpengaruh besar pada convergence

2. **Fairness Metrics (B1, B2, B3)**
   - **B1 (Accuracy Variance)**: Mengukur seberapa seragam akurasi antar klien
   - **B2 (Gini)**: Mengukur ketidakmerataan distribusi akurasi (0=perfectly fair)
   - **B3 (Participation Fairness)**: Mengukur seberapa merata pemilihan klien per round

3. **Perbedaan Strategi**
   - **Random**: Selection sepenuhnya acak → B3 sedang (natural variance)
   - **Performance**: Bias terhadap fast clients → B3 lebih tinggi (unfair)
   - **Fairness**: Dirancang minimize B3 → B3 terendah (most fair)

### Untuk 54-Experiment Analysis:

✓ Gunakan baseline ini sebagai reference point  
✓ Jalankan semua 27 eksperimen MNIST (3 strategies × 3 alphas × 3 seeds)  
✓ Buat comparison plot: B3 vs Alpha untuk setiap strategi  
✓ Identifikasi alpha mana yang menunjukkan perbedaan fairness paling signifikan  
✓ Lalu analisis CIFAR-10 untuk melihat apakah pola sama pada dataset yang lebih kompleks  

### Metrik untuk Fokus di Analysis Akhir:

- **A1 vs Alpha**: Apakah convergence lebih sulit di alpha kecil?
- **B3 vs Alpha**: Apakah perbedaan fairness antar strategi semakin besar di alpha kecil?
- **Performance vs Fairness trade-off**: Apakah fairness strategy sacrifices accuracy?